# Clean Advanced Contextual Bandit Evaluation with Binary and Ordinal Rewards

This notebook evaluates Linear Thompson Sampling and LassoLinUCB on the serious V2 feature sets. It runs the original binary reward and the ordinal bucket reward side by side, while keeping exact bucket accuracy as the main leaderboard metric.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd


def find_repo_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path] + list(path.parents):
        if (candidate / 'src').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not find repository root.')


REPO_ROOT = find_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.bandits.lasso_bandit import LassoLinUCB
from src.bandits.linear_thompson_sampling import LinearThompsonSampling
from src.evaluation.advanced_bandit_benchmark import (
    DEFAULT_LASSO_GRID,
    DEFAULT_TS_GRID,
    FINAL_ADVANCED_FEATURE_SETS,
    REWARD_SCHEMES_TO_COMPARE,
    classwise_advanced_metrics,
    combine_advanced_results,
    evaluate_advanced_grid_for_rewards,
    evaluate_lasso_grid,
    evaluate_linear_ts_grid,
    make_static_baselines_for_feature_sets,
    plot_advanced_leaderboard,
    save_advanced_benchmark,
    summarize_advanced_errors,
)
from src.evaluation.feature_set_benchmark import available_final_feature_sets, load_v2_assets

RESULTS_DIR = REPO_ROOT / 'results' / 'v2_advanced_bandits'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('Repo root:', REPO_ROOT)
print('Results dir:', RESULTS_DIR)

## 1. Load cleaned V2 assets and choose feature sets

In [ ]:
df_v2, feature_sets, preprocess_output_dir = load_v2_assets(REPO_ROOT)
selected_feature_sets = available_final_feature_sets(feature_sets, FINAL_ADVANCED_FEATURE_SETS)

print('V2 modeling table:', df_v2.shape)
print('Selected feature sets:', selected_feature_sets)
print('Reward schemes:', REWARD_SCHEMES_TO_COMPARE)

In [ ]:
static_baselines = make_static_baselines_for_feature_sets(df_v2, selected_feature_sets)
static_baselines.to_csv(RESULTS_DIR / 'static_baselines_for_selected_feature_sets.csv', index=False)
static_baselines.sort_values(['feature_set', 'accuracy'], ascending=[True, False])

## 2. Hyperparameter grids

In [ ]:
print('LinearTS grid:')
for row in DEFAULT_TS_GRID:
    print(row)

print('
LassoLinUCB grid:')
for row in DEFAULT_LASSO_GRID:
    print(row)

## 3. Linear Thompson Sampling under both reward schemes

In [ ]:
SEEDS = range(3)

ts_summary, ts_results, ts_scale_stats = evaluate_advanced_grid_for_rewards(
    evaluator=evaluate_linear_ts_grid,
    df=df_v2,
    feature_sets=feature_sets,
    bandit_cls=LinearThompsonSampling,
    feature_set_names=selected_feature_sets,
    param_grid=DEFAULT_TS_GRID,
    reward_schemes=REWARD_SCHEMES_TO_COMPARE,
    seeds=SEEDS,
    standardize=True,
    progress=False,
    verbose=True,
)

ts_summary.head(15)

## 4. LassoLinUCB under both reward schemes

In [ ]:
lasso_summary, lasso_results, lasso_scale_stats = evaluate_advanced_grid_for_rewards(
    evaluator=evaluate_lasso_grid,
    df=df_v2,
    feature_sets=feature_sets,
    bandit_cls=LassoLinUCB,
    feature_set_names=selected_feature_sets,
    param_grid=DEFAULT_LASSO_GRID,
    reward_schemes=REWARD_SCHEMES_TO_COMPARE,
    seeds=SEEDS,
    standardize=True,
    progress=False,
    verbose=True,
)

lasso_summary.head(15)

## 5. Combined advanced leaderboard

In [ ]:
advanced_summary = pd.concat([ts_summary, lasso_summary], ignore_index=True).sort_values(
    'final_accuracy_mean', ascending=False
)
advanced_results = combine_advanced_results(ts_results, lasso_results)
advanced_scale_stats = pd.concat([ts_scale_stats, lasso_scale_stats], ignore_index=True)

advanced_summary.head(25)

In [ ]:
written = save_advanced_benchmark(
    RESULTS_DIR,
    prefix='advanced_bandit',
    summary=advanced_summary,
    results=advanced_results,
    scale_stats=advanced_scale_stats,
)
written

## 6. Diagnostics for the best advanced configuration

In [ ]:
advanced_error_summary = summarize_advanced_errors(advanced_results)
advanced_error_summary.sort_values('accuracy_mean', ascending=False).head(20)

In [ ]:
advanced_classwise = classwise_advanced_metrics(advanced_results)
best = advanced_summary.iloc[0]
best_filter = (
    (advanced_classwise['feature_set'] == best['feature_set'])
    & (advanced_classwise['algorithm_family'] == best['algorithm_family'])
    & (advanced_classwise['config_name'] == best['config_name'])
    & (advanced_classwise['reward_scheme'] == best['reward_scheme'])
)
advanced_classwise.loc[best_filter].sort_values('class_id')

In [ ]:
fig, ax = plot_advanced_leaderboard(
    advanced_summary,
    top_n=16,
    title='Clean V2 advanced contextual bandits: binary vs ordinal reward',
)
fig.savefig(RESULTS_DIR / 'advanced_bandit_reward_leaderboard.png', dpi=180, bbox_inches='tight')
plt.show()

## 7. Reward comparison checkpoint

The ordinal reward changes what the online model learns from mistakes, but the leaderboard still uses exact bucket accuracy. Check class-wise recall and severe-error rates before deciding whether ordinal reward is genuinely better.

In [ ]:
comparison_cols = [
    'algorithm_family', 'reward_scheme', 'feature_set', 'config_name',
    'final_accuracy_mean', 'final_accuracy_ci95', 'final_reward_mean',
    'late_accuracy_mean', 'final_regret_mean',
]
advanced_summary[comparison_cols].sort_values('final_accuracy_mean', ascending=False).head(30)